In [ ]:
!pip install pandas #libreria per la gestione dei file CSV

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 50.9 MB/s  0:00:00m0:00:01


In [1]:
import numpy as np
import scipy
from scipy import sparse
from scipy.sparse import linalg
import sklearn.utils.extmath
import pandas as pd

# =====================================================================
# FUNZIONE 1: COSTRUZIONE AUTOMATICA DI 'H' e 'a' (OTTIMIZZATA)
# =====================================================================
def prepara_strutture_sparse(sorgenti, destinazioni, n_nodi):
    """
    Costruisce la matrice sparsa H e il vettore 'a' a partire da una semplice 
    lista di link (sorgenti e destinazioni), senza mai creare matrici dense.
    """
    # 1. Creiamo un array di "1" che rappresenta l'esistenza dei link 
    dati_link = np.ones(len(sorgenti))
    
    # 2. Creiamo la Matrice di Adiacenza (A) in formato sparso (CSC - Compressed Sparse Column)
    A = sparse.csc_matrix((dati_link, (destinazioni, sorgenti)), shape=(n_nodi, n_nodi))
    
    # 3. Calcoliamo quanti link escono da ogni pagina (sommando i valori lungo le colonne)
    link_uscenti = A.sum(axis=0).A1 
    
    # 4. Creiamo il vettore 'a' dei dangling nodes in automatico!
    a = (link_uscenti == 0).astype(int)
    
    # 5. Prepariamoci a dividere per calcolare le probabilità (creiamo H)
    link_uscenti_sicuri = np.where(link_uscenti == 0, 1.0, link_uscenti)
    
    # Dividiamo le colonne della matrice sparsa A per il numero di link uscenti.
    diagonale_inversi = sparse.diags(1.0 / link_uscenti_sicuri)
    H = sklearn.utils.extmath.safe_sparse_dot(A, diagonale_inversi)
    
    return H, a 

# ================================================================================
# FUNZIONE 2: L'ALGORITMO PAGERANK MATRIX-FREE
# ===============================================================================
def pagerank_matrix_free(H, a, alpha=0.85, tol=1e-8, max_iter=100):
    """ Calcola il PageRank usando l'approccio Matrix-Free su strutture sparse. """
    n = H.shape[0] # n è inizializzato con il numero di nodi
    
    # 1. Inizializzazione: distribuzione di importanza uniforme (vettore e/n)
    v = np.ones(n) / n 
    e = np.ones(n) 

    # Metodo delle potenze : Iterazione fino alla convergenza o al raggiungimento del numero massimo di iterazioni
    for k in range(1, max_iter + 1):
        # 2. Prodotto scalare v^T * a (gestione della probabilità "persa" nei dangling nodes)
        dangling_sum = v.dot(a)

        # 3. Prodotto matrice-vettore H * v 
        hyperlink_sum = sklearn.utils.extmath.safe_sparse_dot(H, v)

        # 4. Aggiornamento Matrix-Free
        v_new = alpha * hyperlink_sum + (alpha * dangling_sum + (1 - alpha)) * e / n
        
        # 5. Criterio di arresto: controlliamo se la differenza scende sotto la tolleranza
        if np.linalg.norm(v_new - v, 1) < tol:
            print(f"Convergenza raggiunta con successo all'iterazione {k}.")
            return v_new
            
        # Se la convergenza non è ancora raggiunta, aggiorniamo v per la prossima iterazione
        v = v_new
        
    print("Attenzione: convergenza non raggiunta entro il limite massimo di iterazioni.")
    return v


In [ ]:
import os # libreria per far interagire Python con le cartelle del PC

# =====================================================================
# TEST CON I DATASET 100x100 (SCENARIO 1)
# =====================================================================

numero_totale_nodi = 100 
print("--- ANALISI SCENARIO 1 ---")

# 1. Definiamo il percorso del file CSV da leggere
nome_file = '../DataSet_CasoStudio1/Rete_100/dataset_scenario1.csv'
print(f"Caricamento del dataset: {nome_file}...")

# 2. Leggiamo il CSV con Pandas
df = pd.read_csv(nome_file)
nodi_sorgente = df['Source'].values
nodi_destinazione = df['Target'].values

# Generiamo automaticamente le strutture sparse (H e a) senza creare matrici dense
print("Generazione automatica di H sparsa e del vettore a...")
H_sparsa, vettore_a = prepara_strutture_sparse(nodi_sorgente, nodi_destinazione, numero_totale_nodi)

# Esecuzione dell'algoritmo
print("\nAvvio del calcolo del PageRank Matrix-Free...")
pagerank_vector = pagerank_matrix_free(H_sparsa, vettore_a, alpha=0.85)

# Output dei risultati
print("\nRisultato del Ranking (Prime 10 pagine per importanza):")

# Creiamo una lista di tuple (nodo, score_pagerank) e la ordiniamo dal più grande al più piccolo
classifica = [(i, pagerank_vector[i]) for i in range(numero_totale_nodi)]
classifica.sort(key=lambda x: x[1], reverse=True)

# Stampiamo solo i primi 10
for posizione, (nodo, pr) in enumerate(classifica[:10]):
    percentuale = round(pr * 100, 2)
    pr_arrotondato = round(pr, 4)
    print(f"{posizione + 1}° Posto -> Nodo {nodo}: PageRank {pr_arrotondato} ({percentuale}%)")

print("\nVerifica Normalizzazione (Somma probabilità): ", round(np.sum(pagerank_vector), 4))


# =====================================================================
# SALVATAGGIO DEI RISULTATI NELLA CARTELLA 'Risultati_CasoStudio1'
# =====================================================================
# 1. Definiamo il percorso della nuova cartella 
percorso_risultati = '../Risultati_CasoStudio1'

# 2. Se la cartella non esiste, chiediamo al sistema operativo (os) di crearla
if not os.path.exists(percorso_risultati):
    os.makedirs(percorso_risultati)
    print(f"\nCartella creata con successo in: {percorso_risultati}")

# 3. Trasformiamo la classifica in formato tabulare (DataFrame)
df_risultati = pd.DataFrame(classifica, columns=['Nodo', 'PageRank'])

# Aggiungiamo una nuova colonna con la Percentuale calcolata e arrotondata a 2 cifre
df_risultati['Percentuale (%)'] = (df_risultati['PageRank'] * 100).round(2)

# Arrotondiamo la colonna PageRank a 4 cifre decimali 
df_risultati['PageRank'] = df_risultati['PageRank'].round(4)

# 4. Uniamo il percorso della cartella al nome del file
nome_file_risultato = os.path.join(percorso_risultati, 'Classifica_Completa_Scenario1.csv')

# 5. Salviamo il CSV (index=False evita che Pandas aggiunga una colonna inutile di indici numerici)
df_risultati.to_csv(nome_file_risultato, index=False) 

print(f"\nFile salvato correttamente in: {nome_file_risultato}")

--- ANALISI SCENARIO 1 ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_100/dataset_scenario1.csv...
Generazione automatica di H sparsa e del vettore a...

Avvio del calcolo del PageRank Matrix-Free...
Convergenza raggiunta con successo all'iterazione 99.

Risultato del Ranking (Prime 10 pagine per importanza):
1° Posto -> Nodo 0: PageRank 0.1561 (15.61%)
2° Posto -> Nodo 1: PageRank 0.1561 (15.61%)
3° Posto -> Nodo 2: PageRank 0.1561 (15.61%)
4° Posto -> Nodo 3: PageRank 0.0055 (0.55%)
5° Posto -> Nodo 4: PageRank 0.0055 (0.55%)
6° Posto -> Nodo 5: PageRank 0.0055 (0.55%)
7° Posto -> Nodo 6: PageRank 0.0055 (0.55%)
8° Posto -> Nodo 7: PageRank 0.0055 (0.55%)
9° Posto -> Nodo 8: PageRank 0.0055 (0.55%)
10° Posto -> Nodo 9: PageRank 0.0055 (0.55%)

Verifica Normalizzazione (Somma probabilità):  1.0

File salvato correttamente in: ../Risultati_CasoStudio1/Classifica_Completa_Scenario1.csv


In [ ]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST CON I DATASET 1.000.000x1.000.000 (SCENARIO 1)
# =====================================================================

numero_totale_nodi = 1000000 
print("--- ANALISI SCENARIO 1 (1 MILIONE DI NODI) ---")

# 1. PERCORSI 
nome_file_input = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv'
print(f"Caricamento del dataset: {nome_file_input}...")

# 2. Caricamento Dataset e Generazione Strutture
df = pd.read_csv(nome_file_input)
H_sparsa, vettore_a = prepara_strutture_sparse(df['Source'].values, df['Target'].values, numero_totale_nodi)

# 3. Esecuzione dell'algoritmo
print("\nAvvio del calcolo del PageRank Matrix-Free...")
pagerank_vector = pagerank_matrix_free(H_sparsa, vettore_a, alpha=0.85, max_iter=200)

print("\nRisultato del Ranking (Prime 10 pagine per importanza):")

# =====================================================================
# CREAZIONE DATAFRAME OTTIMIZZATA PER BIG DATA 
# =====================================================================
print("Generazione veloce della classifica...")

df_risultati = pd.DataFrame({
    'Nodo': np.arange(numero_totale_nodi), 
    'PageRank': pagerank_vector,
    'Percentuale (%)': (pagerank_vector * 100)
})

# Ordiniamo tutto il dataframe dal più grande al più piccolo in base al PageRank
df_risultati = df_risultati.sort_values(by='PageRank', ascending=False)

# Stampiamo a schermo con la massima precisione scientifica (.4e e .6f)
for posizione, (indice, row) in enumerate(df_risultati.head(10).iterrows()):
    print(f"{posizione + 1}° Posto -> Nodo {int(row['Nodo'])}: PageRank {row['PageRank']:.4e} ({row['Percentuale (%)']:.6f}%)")

print("\nVerifica Normalizzazione (Somma probabilità): ", round(np.sum(pagerank_vector), 4))

# =====================================================================
# SALVATAGGIO DEI RISULTATI NEL CSV (Massima precisione)
# =====================================================================
percorso_risultati = '../Risultati_CasoStudio1_PageRank/Rete_1M/' 
os.makedirs(percorso_risultati, exist_ok=True)

# Arrotondiamo a 8 cifre per i decimali puri e 6 cifre per le percentuali
df_risultati['PageRank'] = df_risultati['PageRank'].round(8) 
df_risultati['Percentuale (%)'] = df_risultati['Percentuale (%)'].round(6)

nome_file_output = 'Classifica_Completa_Scenario1_1MILIONE.csv'
percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati.to_csv(percorso_completo_output, index=False) 

print(f"\nFile salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 1 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario1_1MILIONE.csv...

Avvio del calcolo del PageRank Matrix-Free...
Convergenza raggiunta con successo all'iterazione 118.

Risultato del Ranking (Prime 10 pagine per importanza):
Generazione veloce della classifica...
1° Posto -> Nodo 0: PageRank 1.5315e-01 (15.315344%)
2° Posto -> Nodo 2: PageRank 1.5315e-01 (15.315344%)
3° Posto -> Nodo 1: PageRank 1.5315e-01 (15.315344%)
4° Posto -> Nodo 999998: PageRank 5.4054e-07 (0.000054%)
5° Posto -> Nodo 666656: PageRank 5.4054e-07 (0.000054%)
6° Posto -> Nodo 666658: PageRank 5.4054e-07 (0.000054%)
7° Posto -> Nodo 666659: PageRank 5.4054e-07 (0.000054%)
8° Posto -> Nodo 666660: PageRank 5.4054e-07 (0.000054%)
9° Posto -> Nodo 666661: PageRank 5.4054e-07 (0.000054%)
10° Posto -> Nodo 666662: PageRank 5.4054e-07 (0.000054%)

Verifica Normalizzazione (Somma probabilità):  1.0

File salvato correttamente in: ../Risultati_CasoStudi

In [9]:
# =====================================================================
# TEST SCENARIO 2: RETE MISTA con 100x100
# =====================================================================

print("--- ANALISI SCENARIO 2 ---")
nome_file_2 = '../DataSet_CasoStudio1/Rete_100/dataset_scenario2.csv'

numero_totale_nodi = 100

# 1. Stampa del caricamento
print(f"Caricamento del dataset: {nome_file_2}...")

# Leggiamo il CSV
#sorgenti_2 e destinazioni_2 sono array numpy che contengono rispettivamente i nodi di partenza e di arrivo dei link presenti nel dataset dello scenario 2
df_2 = pd.read_csv(nome_file_2)
sorgenti_2 = df_2['Source'].values
destinazioni_2 = df_2['Target'].values

# Generiamo automaticamente le strutture sparse (H e a) senza creare matrici dense
print("Generazione automatica di H sparsa e del vettore a...")
H_2, a_2 = prepara_strutture_sparse(sorgenti_2, destinazioni_2, numero_totale_nodi) 

# Esecuzione (NOTA: qui inseriamo max_iter=1000 per farlo convergere, altrimenti potrebbe non convergere con 100 iterazioni)
print("\nAvvio del calcolo del PageRank Matrix-Free...")
pagerank_vector_2 = pagerank_matrix_free(H_2, a_2, alpha=0.85, max_iter=1000)

# Output
print("\nRisultato del Ranking SCENARIO 2 (Top 10):")

# Creiamo una lista di tuple (nodo, score_pagerank) e la ordiniamo dal più grande al più piccolo
classifica_2 = [(i, pagerank_vector_2[i]) for i in range(100)]
classifica_2.sort(key=lambda x: x[1], reverse=True) 

# Stampiamo solo i primi 10
for posizione, (nodo, pr) in enumerate(classifica_2[:10]):
    percentuale = round(pr * 100, 2)
    pr_arrotondato = round(pr, 4)
    print(f"{posizione + 1}° Posto -> Nodo {nodo}: PageRank {pr_arrotondato} ({percentuale}%)") 

print("\nVerifica Normalizzazione (Somma probabilità): ", round(np.sum(pagerank_vector_2), 4))


# =====================================================================
# SALVATAGGIO DEI RISULTATI NELLA CARTELLA 'Risultati_CasoStudio1'
# =====================================================================
percorso_risultati = '../Risultati_CasoStudio1_PageRank/Rete_100/'

# Se la cartella non esiste, creiamola (nel caso partissi direttamente da questa cella)
if not os.path.exists(percorso_risultati):
    os.makedirs(percorso_risultati)

# Trasformiamo la classifica in formato tabulare (DataFrame)
df_risultati_2 = pd.DataFrame(classifica_2, columns=['Nodo', 'PageRank'])

# Aggiungiamo una nuova colonna con la Percentuale calcolata e arrotondata a 2 cifre
df_risultati_2['Percentuale (%)'] = (df_risultati_2['PageRank'] * 100).round(2)

# Arrotondiamo la colonna PageRank a 4 cifre decimali 
df_risultati_2['PageRank'] = df_risultati_2['PageRank'].round(4)

# Uniamo il percorso della cartella al nome del file (ATTENZIONE: Scenario 2)
nome_file_risultato_2 = os.path.join(percorso_risultati, 'Classifica_Completa_Scenario2.csv')

# Salviamo il CSV
df_risultati_2.to_csv(nome_file_risultato_2, index=False) 

print(f"\nFile salvato correttamente in: {nome_file_risultato_2}")

--- ANALISI SCENARIO 2 ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_100/dataset_scenario2.csv...
Generazione automatica di H sparsa e del vettore a...

Avvio del calcolo del PageRank Matrix-Free...
Convergenza raggiunta con successo all'iterazione 105.

Risultato del Ranking SCENARIO 2 (Top 10):
1° Posto -> Nodo 1: PageRank 0.2588 (25.88%)
2° Posto -> Nodo 2: PageRank 0.2526 (25.26%)
3° Posto -> Nodo 0: PageRank 0.2475 (24.75%)
4° Posto -> Nodo 61: PageRank 0.0045 (0.45%)
5° Posto -> Nodo 71: PageRank 0.0043 (0.43%)
6° Posto -> Nodo 91: PageRank 0.0043 (0.43%)
7° Posto -> Nodo 81: PageRank 0.0043 (0.43%)
8° Posto -> Nodo 72: PageRank 0.004 (0.4%)
9° Posto -> Nodo 92: PageRank 0.0039 (0.39%)
10° Posto -> Nodo 62: PageRank 0.0039 (0.39%)

Verifica Normalizzazione (Somma probabilità):  1.0

File salvato correttamente in: ../Risultati_CasoStudio1_PageRank/Rete_100/Classifica_Completa_Scenario2.csv


In [7]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST SCENARIO 2: RETE MISTA con 1.000.000x1.000.000
# =====================================================================

numero_totale_nodi = 1000000 
print("--- ANALISI SCENARIO 2 (1 MILIONE DI NODI) ---")

# 1. PERCORSI CORRETTI (Input)
nome_file_input = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv'
print(f"Caricamento del dataset: {nome_file_input}...")

# 2. Caricamento Dataset e Generazione Strutture
df = pd.read_csv(nome_file_input)
H_sparsa, vettore_a = prepara_strutture_sparse(df['Source'].values, df['Target'].values, numero_totale_nodi)

# 3. Esecuzione dell'algoritmo
print("\nAvvio del calcolo del PageRank Matrix-Free...")
pagerank_vector = pagerank_matrix_free(H_sparsa, vettore_a, alpha=0.85, max_iter=1000)

print("\nRisultato del Ranking (Prime 10 pagine per importanza):")

# =====================================================================
# CREAZIONE DATAFRAME OTTIMIZZATA PER BIG DATA
# =====================================================================
print("Generazione veloce della classifica...")

df_risultati = pd.DataFrame({
    'Nodo': np.arange(numero_totale_nodi), 
    'PageRank': pagerank_vector,
    'Percentuale (%)': (pagerank_vector * 100)
})

# Ordiniamo tutto il dataframe dal più grande al più piccolo in base al PageRank
df_risultati = df_risultati.sort_values(by='PageRank', ascending=False)

# Stampiamo a schermo con la massima precisione scientifica (.4e e .6f)
for posizione, (indice, row) in enumerate(df_risultati.head(10).iterrows()):
    print(f"{posizione + 1}° Posto -> Nodo {int(row['Nodo'])}: PageRank {row['PageRank']:.4e} ({row['Percentuale (%)']:.6f}%)")

print("\nVerifica Normalizzazione (Somma probabilità): ", round(np.sum(pagerank_vector), 4))

# =====================================================================
# SALVATAGGIO DEI RISULTATI NEL CSV (Massima precisione)
# =====================================================================
percorso_risultati = '../Risultati_CasoStudio1_PageRank/Rete_1M/' 
os.makedirs(percorso_risultati, exist_ok=True)

# Arrotondiamo a 8 cifre per i decimali puri e 6 cifre per le percentuali
df_risultati['PageRank'] = df_risultati['PageRank'].round(8) 
df_risultati['Percentuale (%)'] = df_risultati['Percentuale (%)'].round(6)

nome_file_output = 'Classifica_Completa_Scenario2_1MILIONE.csv'
percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati.to_csv(percorso_completo_output, index=False) 

print(f"\nFile salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 2 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario2_1MILIONE.csv...

Avvio del calcolo del PageRank Matrix-Free...
Convergenza raggiunta con successo all'iterazione 18.

Risultato del Ranking (Prime 10 pagine per importanza):
Generazione veloce della classifica...
1° Posto -> Nodo 0: PageRank 2.7041e-01 (27.040638%)
2° Posto -> Nodo 1: PageRank 2.7041e-01 (27.040638%)
3° Posto -> Nodo 2: PageRank 2.7041e-01 (27.040638%)
4° Posto -> Nodo 926685: PageRank 6.0929e-07 (0.000061%)
5° Posto -> Nodo 803710: PageRank 5.9972e-07 (0.000060%)
6° Posto -> Nodo 728546: PageRank 5.9468e-07 (0.000059%)
7° Posto -> Nodo 814422: PageRank 5.8142e-07 (0.000058%)
8° Posto -> Nodo 944144: PageRank 5.7732e-07 (0.000058%)
9° Posto -> Nodo 509260: PageRank 5.7698e-07 (0.000058%)
10° Posto -> Nodo 939872: PageRank 5.7408e-07 (0.000057%)

Verifica Normalizzazione (Somma probabilità):  1.0

File salvato correttamente in: ../Risultati_CasoStudio

In [ ]:
# =====================================================================
# TEST SCENARIO 3: COMMUNITY ORGANICA E ATTIVA con 100x100
# =====================================================================

print("--- ANALISI SCENARIO 3 ---")
nome_file_3 = '../DataSet_CasoStudio1/Rete_100/dataset_scenario3.csv'
numero_totale_nodi = 100

# Leggiamo il CSV
#sorgenti_3 e destinazioni_3 sono array numpy che contengono rispettivamente i nodi di partenza e di arrivo dei link presenti nel dataset dello scenario 3
df_3 = pd.read_csv(nome_file_3)
sorgenti_3 = df_3['Source'].values
destinazioni_3 = df_3['Target'].values


# Generiamo automaticamente le strutture sparse (H e a) senza creare matrici dense
print("Generazione automatica di H sparsa e del vettore a...")
H_3, a_3 = prepara_strutture_sparse(sorgenti_3, destinazioni_3, numero_totale_nodi)

# Esecuzione (NOTA: qui inseriamo max_iter=1000 per farlo convergere, altrimenti potrebbe non convergere con 100 iterazioni)
print("\nAvvio del calcolo del PageRank Matrix-Free...")
pagerank_vector_3 = pagerank_matrix_free(H_3, a_3, alpha=0.85, max_iter=1000)

# Output dei risultati
print("\nRisultato del Ranking SCENARIO 3 (Top 10):")

# Creiamo una lista di tuple (nodo, score_pagerank) e la ordiniamo dal più grande al più piccolo
classifica_3 = [(i, pagerank_vector_3[i]) for i in range(100)]
classifica_3.sort(key=lambda x: x[1], reverse=True)

# Stampiamo solo i primi 10
for posizione, (nodo, pr) in enumerate(classifica_3[:10]):
    percentuale = round(pr * 100, 2)
    pr_arrotondato = round(pr, 4)
    print(f"{posizione + 1}° Posto -> Nodo {nodo}: PageRank {pr_arrotondato} ({percentuale}%)")

print("\nVerifica Normalizzazione (Somma probabilità): ", round(np.sum(pagerank_vector_3), 4)) 

# =====================================================================
# SALVATAGGIO DEI RISULTATI NELLA CARTELLA 'Risultati_CasoStudio1'
# =====================================================================
percorso_risultati = '../Risultati_CasoStudio1'

# Se la cartella non esiste, creiamola
if not os.path.exists(percorso_risultati):
    os.makedirs(percorso_risultati)

# Trasformiamo la classifica in formato tabulare (DataFrame)
df_risultati_3 = pd.DataFrame(classifica_3, columns=['Nodo', 'PageRank'])

# Aggiungiamo una nuova colonna con la Percentuale calcolata e arrotondata a 2 cifre
df_risultati_3['Percentuale (%)'] = (df_risultati_3['PageRank'] * 100).round(2)

# Arrotondiamo la colonna PageRank a 4 cifre decimali 
df_risultati_3['PageRank'] = df_risultati_3['PageRank'].round(4)

# Uniamo il percorso della cartella al nome del file (ATTENZIONE: Scenario 3)
nome_file_risultato_3 = os.path.join(percorso_risultati, 'Classifica_Completa_Scenario3.csv')

# Salviamo il CSV
df_risultati_3.to_csv(nome_file_risultato_3, index=False) 

print(f"\nFile salvato correttamente in: {nome_file_risultato_3}")

--- ANALISI SCENARIO 3 ---
Generazione automatica di H sparsa e del vettore a...

Avvio del calcolo del PageRank Matrix-Free...
Convergenza raggiunta con successo all'iterazione 42.

Risultato del Ranking SCENARIO 3 (Top 10):
1° Posto -> Nodo 1: PageRank 0.1031 (10.31%)
2° Posto -> Nodo 0: PageRank 0.0939 (9.39%)
3° Posto -> Nodo 2: PageRank 0.075 (7.5%)
4° Posto -> Nodo 45: PageRank 0.0604 (6.04%)
5° Posto -> Nodo 80: PageRank 0.0562 (5.62%)
6° Posto -> Nodo 55: PageRank 0.0489 (4.89%)
7° Posto -> Nodo 25: PageRank 0.0373 (3.73%)
8° Posto -> Nodo 10: PageRank 0.0292 (2.92%)
9° Posto -> Nodo 15: PageRank 0.0282 (2.82%)
10° Posto -> Nodo 95: PageRank 0.0239 (2.39%)

Verifica Normalizzazione (Somma probabilità):  1.0

File salvato correttamente in: ../Risultati_CasoStudio1/Classifica_Completa_Scenario3.csv


In [ ]:
import os
import pandas as pd
import numpy as np

# =====================================================================
# TEST SCENARIO 3: COMMUNITY ORGANICA E ATTIVA con 1.000.000x1.000.000
# =====================================================================

numero_totale_nodi = 1000000 
print("--- ANALISI SCENARIO 3 (1 MILIONE DI NODI) ---")

# 1. PERCORSI
nome_file_input = '../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv'
print(f"Caricamento del dataset: {nome_file_input}...")

# 2. Caricamento Dataset e Generazione Strutture
df = pd.read_csv(nome_file_input)
H_sparsa, vettore_a = prepara_strutture_sparse(df['Source'].values, df['Target'].values, numero_totale_nodi)

# 3. Esecuzione dell'algoritmo
print("\nAvvio del calcolo del PageRank Matrix-Free...")
pagerank_vector = pagerank_matrix_free(H_sparsa, vettore_a, alpha=0.85, max_iter=1000)

print("\nRisultato del Ranking (Prime 10 pagine per importanza):")

# =====================================================================
# CREAZIONE DATAFRAME OTTIMIZZATA PER BIG DATA
# =====================================================================
print("Generazione veloce della classifica...")

df_risultati = pd.DataFrame({
    'Nodo': np.arange(numero_totale_nodi), 
    'PageRank': pagerank_vector,
    'Percentuale (%)': (pagerank_vector * 100)
})

# Ordiniamo tutto il dataframe dal più grande al più piccolo in base al PageRank
df_risultati = df_risultati.sort_values(by='PageRank', ascending=False)

# Stampiamo a schermo con la massima precisione scientifica (.4e e .6f)
for posizione, (indice, row) in enumerate(df_risultati.head(10).iterrows()):
    print(f"{posizione + 1}° Posto -> Nodo {int(row['Nodo'])}: PageRank {row['PageRank']:.4e} ({row['Percentuale (%)']:.6f}%)")

print("\nVerifica Normalizzazione (Somma probabilità): ", round(np.sum(pagerank_vector), 4))

# =====================================================================
# SALVATAGGIO DEI RISULTATI NEL CSV (Massima precisione)
# =====================================================================
percorso_risultati = '../Risultati_CasoStudio1_PageRank/Rete_1M/' 
os.makedirs(percorso_risultati, exist_ok=True)

# Arrotondiamo a 8 cifre per i decimali puri e 6 cifre per le percentuali
df_risultati['PageRank'] = df_risultati['PageRank'].round(8)
df_risultati['Percentuale (%)'] = df_risultati['Percentuale (%)'].round(6)

nome_file_output = 'Classifica_Completa_Scenario3_1MILIONE.csv'
percorso_completo_output = os.path.join(percorso_risultati, nome_file_output)
df_risultati.to_csv(percorso_completo_output, index=False) 

print(f"\nFile salvato correttamente in: {percorso_completo_output}")

--- ANALISI SCENARIO 3 (1 MILIONE DI NODI) ---
Caricamento del dataset: ../DataSet_CasoStudio1/Rete_1M/dataset_scenario3_1MILIONE.csv...

Avvio del calcolo del PageRank Matrix-Free...
Convergenza raggiunta con successo all'iterazione 36.

Risultato del Ranking (Prime 10 pagine per importanza):
Generazione veloce della classifica...
1° Posto -> Nodo 0: PageRank 1.2255e-01 (12.255167%)
2° Posto -> Nodo 1: PageRank 8.9917e-02 (8.991676%)
3° Posto -> Nodo 2: PageRank 6.0927e-02 (6.092722%)
4° Posto -> Nodo 100000: PageRank 1.5756e-03 (0.157559%)
5° Posto -> Nodo 91000: PageRank 1.5697e-03 (0.156965%)
6° Posto -> Nodo 770000: PageRank 1.5584e-03 (0.155838%)
7° Posto -> Nodo 830000: PageRank 1.5584e-03 (0.155837%)
8° Posto -> Nodo 636000: PageRank 1.5561e-03 (0.155610%)
9° Posto -> Nodo 983000: PageRank 1.5555e-03 (0.155554%)
10° Posto -> Nodo 664000: PageRank 1.2400e-03 (0.124004%)

Verifica Normalizzazione (Somma probabilità):  1.0

File salvato correttamente in: ../Risultati_CasoStudio1_P